# 04-03 检索与重排序

**单一检索的局限**：
- 稠密检索（向量）：擅长语义相似，但可能漏掉关键词精确匹配
- 稀疏检索（BM25）：擅长关键词，但不理解语义
- 混合检索 + 重排序 = 最强效果

**本节目标**：BM25 实现、混合检索、Cross-Encoder 重排序、评估指标

---

In [ ]:
import numpy as np
import math
import sys
sys.path.insert(0, "..")
from utils.data_generator import generate_ad_knowledge_base

docs = generate_ad_knowledge_base()
print(f"知识库: {len(docs)} 个文档")

def get_embeddings(texts):
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
        return model.encode(texts, normalize_embeddings=True)
    except ImportError:
        embs = []
        for t in texts:
            rng = np.random.default_rng(hash(t) % (2**32))
            v = rng.standard_normal(128).astype(np.float32)
            embs.append(v / np.linalg.norm(v))
        return np.array(embs)

doc_embeddings = get_embeddings(docs)

## 1. BM25 稀疏检索

In [ ]:
import re
from collections import Counter

class BM25:
    """
    BM25 (Best Match 25) 检索算法
    - 比简单 TF-IDF 更好的词频饱和处理
    - 考虑文档长度归一化
    """
    def __init__(self, docs: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.docs = docs
        self.N = len(docs)
        
        # 分词（简单字符级，中文无空格）
        self.tokenized = [self._tokenize(d) for d in docs]
        self.doc_lengths = [len(t) for t in self.tokenized]
        self.avgdl = sum(self.doc_lengths) / self.N
        
        # 计算 IDF
        self.idf = self._compute_idf()
    
    def _tokenize(self, text: str) -> list[str]:
        """简单分词：中文字符 + 英文单词"""
        # 中文每个字一个 token，英文按空格分
        tokens = []
        for char in text:
            if '\u4e00' <= char <= '\u9fff':
                tokens.append(char)
            elif char.isalnum():
                tokens.append(char.lower())
        return tokens
    
    def _compute_idf(self) -> dict:
        """计算逆文档频率"""
        df = Counter()
        for tokens in self.tokenized:
            for token in set(tokens):
                df[token] += 1
        return {
            token: math.log((self.N - count + 0.5) / (count + 0.5) + 1)
            for token, count in df.items()
        }
    
    def score(self, query: str, doc_idx: int) -> float:
        """计算 query 与 doc 的 BM25 分数"""
        q_tokens = self._tokenize(query)
        doc_tokens = self.tokenized[doc_idx]
        doc_len = self.doc_lengths[doc_idx]
        tf = Counter(doc_tokens)
        
        score = 0.0
        for token in q_tokens:
            if token not in self.idf:
                continue
            freq = tf.get(token, 0)
            # BM25 公式
            numerator = freq * (self.k1 + 1)
            denominator = freq + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
            score += self.idf[token] * numerator / denominator
        return score
    
    def search(self, query: str, k: int = 3) -> list[tuple]:
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:k]

bm25 = BM25(docs)

query = "广告CTR计算方法"
results = bm25.search(query, k=3)
print(f"BM25 检索: {query!r}")
for idx, score in results:
    print(f"  [BM25={score:.3f}] {docs[idx][:60]}...")

## 2. 混合检索（Hybrid Retrieval）

In [ ]:
def dense_search(query: str, doc_embeddings: np.ndarray, k: int = 5) -> list[tuple]:
    """稠密检索（向量相似度）"""
    q_emb = get_embeddings([query])[0]
    sims = doc_embeddings @ q_emb
    top_k = np.argsort(sims)[::-1][:k]
    return [(int(i), float(sims[i])) for i in top_k]


def hybrid_search(
    query: str,
    doc_embeddings: np.ndarray,
    bm25: BM25,
    alpha: float = 0.5,  # 0=纯BM25, 1=纯向量
    k: int = 3
) -> list[tuple]:
    """
    混合检索：线性加权合并稠密 + 稀疏分数
    使用 RRF (Reciprocal Rank Fusion) 或简单归一化加权
    """
    N = len(docs)
    
    # 稠密分数
    dense_results = dense_search(query, doc_embeddings, k=N)
    dense_scores = {idx: score for idx, score in dense_results}
    
    # BM25 分数
    bm25_results = bm25.search(query, k=N)
    bm25_scores = {idx: score for idx, score in bm25_results}
    
    # 归一化到 [0, 1]
    max_dense = max(dense_scores.values()) or 1
    max_bm25 = max(bm25_scores.values()) or 1
    
    # 合并分数
    combined = {}
    for i in range(N):
        d_score = dense_scores.get(i, 0) / max_dense
        b_score = bm25_scores.get(i, 0) / max_bm25
        combined[i] = alpha * d_score + (1 - alpha) * b_score
    
    sorted_results = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    return sorted_results[:k]


# 对比三种检索方式
test_queries = [
    "CTR如何计算",
    "视频广告时长规格",
]

for query in test_queries:
    print(f"\n=== 查询: {query!r} ===")
    
    dense = dense_search(query, doc_embeddings, k=2)
    bm25_res = bm25.search(query, k=2)
    hybrid = hybrid_search(query, doc_embeddings, bm25, alpha=0.6, k=2)
    
    print(f"  稠密检索: {[docs[i][:30]+'...' for i,s in dense]}")
    print(f"  BM25:     {[docs[i][:30]+'...' for i,s in bm25_res]}")
    print(f"  混合:     {[docs[i][:30]+'...' for i,s in hybrid]}")

## 3. 重排序（Reranking）

In [ ]:
import sys
sys.path.insert(0, "..")
from utils.llm_client import call_llm

def llm_rerank(query: str, candidate_docs: list[str], top_k: int = 3) -> list[tuple]:
    """
    用 LLM 对候选文档重排序（LLM as Reranker）
    比 Cross-Encoder 更强但更慢更贵
    """
    candidates_text = "\n".join(
        f"[{i}] {doc[:100]}" for i, doc in enumerate(candidate_docs)
    )
    
    prompt = f"""对于问题「{query}」，从以下候选文档中选出最相关的{top_k}个，
返回文档编号列表（JSON格式），从最相关到最不相关排序。

{candidates_text}

返回格式: {{"ranked": [0, 2, 1]}}  (文档编号，最相关在前)"""
    
    try:
        response = call_llm(prompt, max_tokens=100)
        import json, re
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            data = json.loads(match.group())
            ranked_indices = data.get("ranked", list(range(len(candidate_docs))))
            return [(i, len(candidate_docs) - rank) for rank, i in enumerate(ranked_indices[:top_k])]
    except Exception:
        pass
    
    # Fallback: 原始顺序
    return [(i, len(candidate_docs) - i) for i in range(min(top_k, len(candidate_docs)))]


def keyword_rerank(query: str, docs: list[str], top_k: int = 3) -> list[tuple]:
    """简单关键词重排（演示用，生产环境用 cross-encoder）"""
    query_words = set(query)
    scores = []
    for i, doc in enumerate(docs):
        # 关键词命中率
        hits = sum(1 for w in query_words if w in doc)
        scores.append((i, hits / max(len(query_words), 1)))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]


# 演示完整 retrieve → rerank 流程
query = "B站广告出价方式有哪些"
print(f"查询: {query!r}")

# Step 1: 初检 top-5
initial_results = hybrid_search(query, doc_embeddings, bm25, alpha=0.5, k=5)
candidates = [docs[i] for i, _ in initial_results]
print(f"\n初检 Top-5:")
for i, (doc_idx, score) in enumerate(initial_results):
    print(f"  [{i}] (score={score:.3f}) {docs[doc_idx][:50]}...")

# Step 2: 重排序 → top-3
reranked = keyword_rerank(query, candidates, top_k=3)
print(f"\n重排序后 Top-3:")
for rank, (doc_local_idx, score) in enumerate(reranked):
    doc_global_idx = initial_results[doc_local_idx][0]
    print(f"  [{rank+1}] (rerank_score={score:.3f}) {docs[doc_global_idx][:60]}...")

## 4. 检索评估指标

In [ ]:
def precision_at_k(retrieved: list[int], relevant: set[int], k: int) -> float:
    """P@K: 前K个结果中有多少是相关的"""
    return len(set(retrieved[:k]) & relevant) / k

def recall_at_k(retrieved: list[int], relevant: set[int], k: int) -> float:
    """R@K: 所有相关文档中有多少被检索到"""
    return len(set(retrieved[:k]) & relevant) / max(len(relevant), 1)

def mrr(retrieved: list[int], relevant: set[int]) -> float:
    """MRR: 第一个相关文档的排名倒数均值"""
    for rank, doc_id in enumerate(retrieved, 1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

# 测试集（模拟标注好的 ground truth）
test_cases = [
    {"query": "CTR计算",      "relevant_ids": {0}},
    {"query": "广告审核时间", "relevant_ids": {5}},
    {"query": "投放预算",     "relevant_ids": {3, 4}},
]

print(f"{'查询':<15} {'方法':<10} P@3    R@3    MRR")
print("-" * 55)

for tc in test_cases:
    query = tc["query"]
    relevant = tc["relevant_ids"]
    
    # 稠密检索
    dense_ids = [i for i, _ in dense_search(query, doc_embeddings, k=5)]
    p3_d = precision_at_k(dense_ids, relevant, 3)
    r3_d = recall_at_k(dense_ids, relevant, 3)
    mrr_d = mrr(dense_ids, relevant)
    print(f"{query:<15} {'Dense':<10} {p3_d:.2f}   {r3_d:.2f}   {mrr_d:.2f}")
    
    # 混合检索
    hybrid_ids = [i for i, _ in hybrid_search(query, doc_embeddings, bm25, k=5)]
    p3_h = precision_at_k(hybrid_ids, relevant, 3)
    r3_h = recall_at_k(hybrid_ids, relevant, 3)
    mrr_h = mrr(hybrid_ids, relevant)
    print(f"{'':<15} {'Hybrid':<10} {p3_h:.2f}   {r3_h:.2f}   {mrr_h:.2f}")
    print()

## 面试速记

| 问题 | 要点 |
|------|------|
| BM25 vs 向量检索 | BM25：精确词匹配好；向量：语义理解好；混合最强 |
| Reranking 的必要性 | 初检 recall 高但 precision 低，reranker 提升 precision |
| Cross-Encoder vs Bi-Encoder | Cross: query+doc一起encode，精度高但慢；Bi: 分开encode，快但精度低 |
| RAG 评估指标 | Context Precision（检索是否相关）、Context Recall（相关文档是否都检到）、Answer Faithfulness（答案是否忠实于文档）|

**下一节**: `04_advanced_rag.ipynb`